In [1]:
import pandas as pd
import numpy as np

# Sample product sales data
df = pd.DataFrame({
    'Product': ['A', 'B', 'C', 'D', 'E'],
    'Sales': [100, 150, 80, 200, 120],
    'Profit': [20, 30, 10, 50, 25],
    'Tags': [['new', 'popular'], ['discount'], ['popular'], ['new'], ['clearance', 'discount']]
})


In [2]:
print("\nCorrelation:\n", df[['Sales', 'Profit']].corr())
print("\nCovariance:\n", df[['Sales', 'Profit']].cov())

# correction matrix - measures how strongly two variables move together, scaled between -1 and 1
#what is the correlation value between sales and profit - 0.98, very close to +1, this means as sales increase, profit also increases. Strength and direction of relationship
# high correlation may indicate redundant features, in such cases better to drop one to reduce multicollinearity in models

# covariance - measures how two variables vary together
# sales-sales(2200) - this is variance of sales - how spread out the sales values are.
# profit - profit - this is variance of the profit
# sales-profit (687.5) - covariance between sales and profit, positive covariance between sales and profit confirms that as sales increase, profit tends to increase
# sales variance is much higher tan profit variance - sales values are more spread out than profitML

# corr(X,Y) = cov(X,Y) / sigma~x . sigma~y

# ML - feature selection (corr), comparing relationship across features (corr)


Correlation:
            Sales    Profit
Sales   1.000000  0.988212
Profit  0.988212  1.000000

Covariance:
          Sales  Profit
Sales   2200.0   687.5
Profit   687.5   220.0


In [3]:
df['SalesRank'] = df['Sales'].rank(ascending=False)
df['ProfitDiff'] = df['Profit'].diff()
print("\nRank & Diff:\n", df[['Product', 'SalesRank', 'ProfitDiff']])

# rank - ranks the products by Sales, with highest sales get rank 1
# diff - cal difference in profit from previous row
#ML - rank - feature scaling, encoding, scoring. diff - analysis w.r.t time, trends


Rank & Diff:
   Product  SalesRank  ProfitDiff
0       A        4.0         NaN
1       B        2.0        10.0
2       C        5.0       -20.0
3       D        1.0        40.0
4       E        3.0       -25.0


In [4]:
df['ClippedProfit'] = df['Profit'].clip(lower=15, upper=40)
df['RoundedSales'] = df['Sales'].round(-1)
print("\nClipped & Rounded:\n", df[['Product', 'ClippedProfit', 'RoundedSales']])
# clip(lower=15, upper=40) - limit the values in "Profit" to stay within the range 15 to 40, that means if any value below 15 becomes 15, any value abve 40 becomes 40
#round(-1) - round sales to nearest multiple of 10 - that means -1 means roudning to tens place
#ML  - clip - control outliers, normalize the data. round - reduce noise


Clipped & Rounded:
   Product  ClippedProfit  RoundedSales
0       A             20           100
1       B             30           150
2       C             15            80
3       D             40           200
4       E             25           120


In [5]:
df_encoded = pd.get_dummies(df['Product'], prefix='Prod')
print("\nOne-Hot Encoded Products:\n", df_encoded)

# basic technique in preparing catergorical data for ML models
# convert categorical variables into certain format where Ml models like linear regression, decisions trees, etc
# Ml - avoid bias, enables compatibility with models this means ML algorithms need numeric input
#HW - try on more than one feature


One-Hot Encoded Products:
    Prod_A  Prod_B  Prod_C  Prod_D  Prod_E
0    True   False   False   False   False
1   False    True   False   False   False
2   False   False    True   False   False
3   False   False   False    True   False
4   False   False   False   False    True


In [ ]:
# above output
#A B C D E
#1 0 0 0 0
#0 1 0 0 0
#0 0 1 0 0
#0 0 0 0 1

In [6]:
df['ProductCode'], product_labels = pd.factorize(df['Product'])
print("\nFactorized Products:\n", df[['Product', 'ProductCode']])
# assigns a unqiue integer cde to each uniquer value for Product feature
#what is the difference between factorize() and get_dummies()?
#integer codes, binary columns
#get dummies is more suitable for linear models


Factorized Products:
   Product  ProductCode
0       A            0
1       B            1
2       C            2
3       D            3
4       E            4


In [7]:
df['SalesBin'] = pd.cut(df['Sales'], bins=[0, 100, 150, 250], labels=['Low', 'Medium', 'High'])
print("\nSales Binned:\n", df[['Product', 'Sales', 'SalesBin']])
#splits the sales values into three bins
#0 - 100 : Low
#100-150 - Medium
#150 - 250 - High


Sales Binned:
   Product  Sales SalesBin
0       A    100      Low
1       B    150   Medium
2       C     80      Low
3       D    200     High
4       E    120   Medium


In [8]:
df['ProfitMargin'] = df.eval('Profit / Sales * 100')
print("\nProfit Margin:\n", df[['Product', 'ProfitMargin']])
#eval - to evaluate string expression


Profit Margin:
   Product  ProfitMargin
0       A     20.000000
1       B     20.000000
2       C     12.500000
3       D     25.000000
4       E     20.833333


In [9]:
df['MaskedProfit'] = df['Profit'].mask(df['Profit'] < 20, np.nan)
df['WhereSales'] = df['Sales'].where(df['Sales'] > 100, 0)
print("\nMasked & Where:\n", df[['Product', 'MaskedProfit', 'WhereSales']])
#


Masked & Where:
   Product  MaskedProfit  WhereSales
0       A          20.0           0
1       B          30.0         150
2       C           NaN           0
3       D          50.0         200
4       E          25.0         120


In [10]:
df_sample = df.sample(n=3, random_state=42)
print("\nRandom Sample:\n", df_sample)
#sample(n=3, random_state=42) , selects 3 random rows from the dataframe, random_state=42 ensures reproducibility


Random Sample:
   Product  Sales  Profit                   Tags  SalesRank  ProfitDiff  \
1       B    150      30             [discount]        2.0        10.0   
4       E    120      25  [clearance, discount]        3.0       -25.0   
2       C     80      10              [popular]        5.0       -20.0   

   ClippedProfit  RoundedSales  ProductCode SalesBin  ProfitMargin  \
1             30           150            1   Medium     20.000000   
4             25           120            4   Medium     20.833333   
2             15            80            2      Low     12.500000   

   MaskedProfit  WhereSales  
1          30.0         150  
4          25.0         120  
2           NaN           0  


In [11]:
df_exploded = df.explode('Tags')
print("\nExploded Tags:\n", df_exploded[['Product', 'Tags']])
# explode to flatten a list like column - multi label feature - when working with keywords


Exploded Tags:
   Product       Tags
0       A        new
0       A    popular
1       B   discount
2       C    popular
3       D        new
4       E  clearance
4       E   discount
